## Nome: Felipe Silva Loschi
## Matricula: 601
-----


# **Atividade 16 - Testes Estatísticos Caracterizados usando a Escala de Dependência de Smartphone**



## Instruções gerais

- Execute as células na ordem em que aparecem.
- Sempre **declare as hipóteses** antes de calcular (H0 = igualdade; H1 = diferença).
- Para tabelas 2×2, você pode usar:
  - `stats.chi2_contingency(obs, correction=False)` → χ² clássico;
  - `stats.chi2_contingency(obs, correction=True)` → com **Yates**;
  - `stats.fisher_exact(obs, alternative="two-sided")` → **Fisher** (n pequeno ou valores esperados < 5);
  - `mcnemar(tabela_pareada, exact=True/False, correction=True/False)` → **McNemar** (amostras pareadas).

In [1]:
# Imports úteis para toda a atividade
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

pd.set_option("display.precision", 4)



## Funções auxiliares


In [2]:
def esperados_from_obs(obs):
    obs = np.asarray(obs, dtype=float)
    row_sum = obs.sum(axis=1, keepdims=True)
    col_sum = obs.sum(axis=0, keepdims=True)
    total = obs.sum()
    return row_sum @ col_sum / total

def decide_from_p(p, alphas=(0.10, 0.05, 0.01)):
    return {a: ("Rejeita H0" if p < a else "Não rejeita H0") for a in alphas}



# Exercício 1 — Qui-quadrado clássico (amostras independentes, n > 40)

**Contexto:** Uma amostra de estudantes de cursos de Engenharia respondeu à *Cell Phone Dependence Scale*. Cada estudante foi classificado em "Leve", "Moderado" ou "Grave". Para uma análise 2×2, definimos:

- Dependência leve  
- Dependência moderada/grave  

Queremos verificar se o **sexo** do estudante está associado ao nível de dependência de smartphones.

A tabela de valores observados (2×2), n = 224:

| Sexo        | Dependência leve | Dependência moderada/grave | Total |
|-------------|------------------|-----------------------------|--------|
| Feminino    | 11               | 53                          | 64     |
| Masculino   | 57               | 103                         | 160    |
| **Total**   | 68               | 156                         | 224    |

1. Formule H0 e H1.  
2. Calcule a **tabela de esperados** e a estatística **χ² calculado** (sem correção).  
3. Obtenha `gl`, **χ² tabelado** para α ∈ {10%,5%,1%} (use SciPy) e o **p-valor**.  
4. Decida e conclua sobre a **associação entre sexo e dependência de smartphones**.


1. Formule H0 e H1:
  - H0: O sexo do estudante é indiferente para o nível de dependência de smartphones.
  - H1: O sexo do estudante não é indiferente para o nível de dependência de smartphones.

In [3]:
# Tabela observada (Exemplo 1)
obs1 = np.array([[11, 53],
                 [57, 103]])
obs1_df = pd.DataFrame(obs1, index=["Feminino","Masculino"], columns=["Dependência leve","Dependência moderada/grave"])


In [4]:
# χ² clássico (sem correção de Yates) e esperados
chi2, p, dof, expected = stats.chi2_contingency(obs1, correction=False)
expected_df = pd.DataFrame(expected, index=obs1_df.index, columns=obs1_df.columns)



In [5]:

# χ² tabelado (crítico) para níveis usuais
alphas = [0.10, 0.05, 0.01]
chi2_crit = {a: stats.chi2.ppf(1 - a, dof) for a in alphas}


In [6]:
print("Tabela de valores observados:")
display(obs1_df)
print("\nTabela de valores esperados:")
display(expected_df)
print("\nEstatística χ² calculada:", chi2)
print("Graus de liberdade:", dof)
print("Valores críticos para α ∈ {10%, 5%, 1%}:")
print(chi2_crit)
print("\np-valor:", p)
print("\nConclusão:")
print(decide_from_p(p, alphas))

Tabela de valores observados:


,Dependência leve,Dependência moderada/grave
Feminino,11,53
Masculino,57,103



Tabela de valores esperados:


,Dependência leve,Dependência moderada/grave
Feminino,19.4286,44.5714
Masculino,48.5714,111.4286



Estatística χ² calculada: 7.350527903469081
Graus de liberdade: 1
Valores críticos para α ∈ {10%, 5%, 1%}:
{0.1: np.float64(2.705543454095404), 0.05: np.float64(3.841458820694124), 0.01: np.float64(6.6348966010212145)}

p-valor: 0.006704306708170397

Conclusão:
{0.1: 'Rejeita H0', 0.05: 'Rejeita H0', 0.01: 'Rejeita H0'}


**Conclusão:**
Dado o resultado do teste estátistico, decide-se pela rejeição de H0. Logo, para esse grupo teste, o sexo do indivíduo influencia na sua dependência em smartphone.

## Exercício 2 — Teste Exato de Fisher (amostras independentes, n < 20 ou esperados < 5)

**Contexto:** Na mesma amostra de estudantes de Engenharia, além da classificação do nível de dependência de smartphones, foram coletadas informações sobre a participação dos pais na vida escolar.  
Queremos investigar se a **participação dos pais na vida escolar** está associada à ocorrência de **dependência grave** de smartphones.

Para isso, consideramos:
- Dois grupos de pais:
  - **Nunca** participam da vida escolar.
  - **Sempre** participam da vida escolar.
- Duas categorias de dependência:
  - **Grave**
  - **Não grave** (Leve ou Moderado).

A tabela de valores observados (2×2) é:

| Pais participam da vida escolar? | Grave | Não grave | Total |
|----------------------------------|:-----:|:---------:|:-----:|
| Nunca                            |   2   |    45     |  47   |
| Sempre                           |   9   |    69     |  78   |
| **Total**                        |  11   |   114     | 125   |

1. Formule H0 e H1 (duas caudas).
2. Aplique o **teste exato de Fisher** (two-sided) e obtenha o **p-valor**.
3. Decida em α = 5% e conclua sobre a associação entre participação dos pais e dependência grave.


1. Formule H0 e H1:
  - H0: A participação dos pais na vida escolar dos filhos é indiferente para a dependência grave de smartphones.
  - H1: A participação dos pais na vida escolar dos filhos não é indiferente para a deoendência grave de smartphones.

In [7]:

obs3 = np.array([[2,45],
                 [9,69]])
oddsratio, p_fisher = stats.fisher_exact(obs3, alternative="two-sided")


In [8]:
print("Tabela de valores observados:")
obs3_df = pd.DataFrame(obs3, index=["Nunca","Sempre"], columns=["Grave","Não grave"])
display(obs3_df)
print("\np-valor:", p_fisher)
print("\nConclusão:")
print(decide_from_p(p_fisher, [0.05]))

Tabela de valores observados:


,Grave,Não grave
Nunca,2,45
Sempre,9,69



p-valor: 0.20611729432572512

Conclusão:
{0.05: 'Não rejeita H0'}


**Conclusão:**
Baseado no resultado do teste estatístico, decide-se, para esse grupo teste, em não rejeitar a hipótese de nulidade, logo, a participação dos pais na vida escolar é indiferente para a dependência grave dos filhos em smartphone.

----
# Parte Teórica

# Responda as perguntas abaixo, utilizando suas próprias palavras!

1- O que significa dizer que duas variáveis são “independentes” no contexto de estatística?
  - Resposta: dizer que duas variáveis são independentes significa dizer que não há associação entre as variáveis, ou seja, conhecer uma não te ajuda em conhecer a outra.

2- Explique por que não se usa correção de Yates quando a amostra é grande.
  - Resposta: porque a aproximação contínua do qui-quadrado já é precisa o suficiente.

3- Por que o Teste Exato de Fisher é indicado quando n < 20 ou quando valores esperados < 5?
  - Resposta: porque, nessas condições, a aproximação do qui-quadrado se torna imprecisa, podendo gerar p-valores incorretos.

4- Qual é o principal motivo do Teste de Fisher ser considerado “exato”?
  - Resposta: porque ele calcula a probabilidade exata dos dados observados sem utilizar aproximações estatísticas.